# 02 — Feature Engineering
### FUT 26 Player Price Dataset

In this notebook, we transform the raw scraped data into a feature‑rich dataset suitable for machine learning.

We will create:
- lag features  
- moving averages  
- volatility indicators  
- calendar features  
- the target variable (next day price)  

These features will be used in Notebook 3 for model training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.features import build_features

## 1. Load Raw Data

We load the raw CSV created by the scraper.

In [2]:
df_raw = pd.read_csv(RAW_DATA_DIR/"fut_prices_raw.csv")
df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw.head()

,player_id,player_name,date,price
0,158023,lionel-messi,2025-09-17 00:00:00+00:00,20500
1,158023,lionel-messi,2025-09-18 00:00:00+00:00,26500
2,158023,lionel-messi,2025-09-19 00:00:00+00:00,26500
3,158023,lionel-messi,2025-09-20 00:00:00+00:00,20500
4,158023,lionel-messi,2025-09-21 00:00:00+00:00,25750


## 2. Sort and Inspect

We sort by player and date to ensure correct time‑series ordering.

In [3]:
df_raw = df_raw.sort_values(["player_name", "date"])
df_raw.head()

,player_id,player_name,date,price
1172,234396,alphonso-davies,2025-09-17 00:00:00+00:00,29000
1173,234396,alphonso-davies,2025-09-18 00:00:00+00:00,35250
1174,234396,alphonso-davies,2025-09-19 00:00:00+00:00,33500
1175,234396,alphonso-davies,2025-09-20 00:00:00+00:00,26250
1176,234396,alphonso-davies,2025-09-21 00:00:00+00:00,25750


## 3. Apply Feature Engineering

We use the `build_features()` function from `features.py`.

This function creates:
- lag features (1, 3, 7 days)
- moving averages (3, 7 days)
- volatility indicators (3, 7 days)
- day of week
- weekend flag
- target variable: next_day_price

In [4]:
df_features = build_features(df_raw)
df_features.head()

,player_id,player_name,date,price,log_price,lag_1,lag_3,lag_7,ma_3,ma_7,pct_change,vol_3,vol_7,day_of_week,is_weekend,next_day_price
9989,570,okocha,2025-09-24,760000,13.541075,13.563197,13.537120,14.808763,13.543144,13.515434,-0.001631,0.002379,0.035495,2,0,13.460265
9990,570,okocha,2025-09-25,701000,13.460265,13.541075,13.525160,13.441546,13.521512,13.518108,-0.005968,0.004390,0.004101,3,0,13.437176
9991,570,okocha,2025-09-26,685000,13.437176,13.460265,13.563197,13.517106,13.479505,13.506689,-0.001715,0.002480,0.003356,4,0,13.468787
9992,570,okocha,2025-09-27,707000,13.468787,13.437176,13.541075,13.482833,13.455409,13.504683,0.002353,0.004161,0.003452,5,1,13.453106
9993,570,okocha,2025-09-28,696000,13.453106,13.468787,13.460265,13.537120,13.453023,13.492681,-0.001164,0.002207,0.002924,6,1,13.445897


## 4. Check Missing Values

After feature engineering, early rows will be dropped due to lag/rolling windows.

In [5]:
df_features.isna().sum()

player_id         0
player_name       0
date              0
price             0
log_price         0
lag_1             0
lag_3             0
lag_7             0
ma_3              0
ma_7              0
pct_change        0
vol_3             0
vol_7             0
day_of_week       0
is_weekend        0
next_day_price    0
dtype: int64

## 5. Feature List

Let’s inspect all engineered features.

In [6]:
df_features.columns

Index(['player_id', 'player_name', 'date', 'price', 'log_price', 'lag_1',
       'lag_3', 'lag_7', 'ma_3', 'ma_7', 'pct_change', 'vol_3', 'vol_7',
       'day_of_week', 'is_weekend', 'next_day_price'],
      dtype='object')

## 6. Visualizing Lag Features

Lag features help the model understand short‑term momentum.

In [7]:
import plotly.express as px

sample_player = df_features["player_name"].unique()[0]
player_data = df_features[df_features["player_name"] == sample_player]

fig = px.line(player_data, x="date", y=["log_price", "lag_1", "lag_3", "lag_7"], title=f"Lag Features — {sample_player}")
fig.update_xaxes(title_text="Date", tickangle=45)
fig.update_yaxes(title_text="Value")
fig.show()

## 7. Visualizing Moving Averages

Moving averages smooth out noise and capture trend direction.

In [8]:
sample_player = df_features["player_name"].unique()[0]
player_data = df_features[df_features["player_name"] == sample_player]

fig = px.line(player_data, x="date", y=["log_price", "ma_3", "ma_7"], title=f"Moving Averages — {sample_player}")
fig.update_xaxes(title_text="Date", tickangle=45)
fig.update_yaxes(title_text="Price")
fig.show()

## 8. Visualizing Volatility

Volatility features help the model understand how “risky” or unstable a player’s price is.

In [9]:
sample_player = df_features["player_name"].unique()[0]
player_data = df_features[df_features["player_name"] == sample_player]

fig = px.line(player_data, x="date", y=["vol_3", "vol_7"], title=f"Volatility Indicators — {sample_player}")
fig.update_xaxes(title_text="Date", tickangle=45)
fig.update_yaxes(title_text="Volatility")
fig.show()

## 9. Calendar Features

Day‑of‑week and weekend flags capture weekly market behavior:
- Thursday rises (rewards)
- Weekend dips (WL sell‑off)

In [10]:
df_features[["day_of_week", "is_weekend"]].head()

,day_of_week,is_weekend
9989,2,0
9990,3,0
9991,4,0
9992,5,1
9993,6,1


## 10. Target Variable

The target is **next_day_price**, created using:

df["next_day_price"] = df["price"].shift(-1)


This aligns tomorrow’s price with today’s features.

In [11]:
df_features[["log_price", "next_day_price"]].head(10)

,log_price,next_day_price
9989,13.541075,13.460265
9990,13.460265,13.437176
9991,13.437176,13.468787
9992,13.468787,13.453106
9993,13.453106,13.445897
9994,13.445897,13.406039
9995,13.406039,13.339088
9996,13.339088,13.384729
9997,13.384729,13.358227
9998,13.358227,13.321216


## 11. Save Processed Dataset

We save the processed dataset for Notebook 3 (modeling).

In [ ]:
df_features.to_csv(f"{PROCESSED_DATA_DIR}/fut_prices_features.csv", index=False)

# 12. Conclusions

We successfully engineered a rich set of features:

### ✔ Lag features  
Capture short‑term momentum.

### ✔ Moving averages  
Capture trend direction.

### ✔ Volatility indicators  
Capture price stability and risk.

### ✔ Calendar features  
Capture weekly market behavior.

### ✔ Target variable  
Aligns tomorrow’s price with today’s features.

This dataset is now ready for model training in **Notebook 3 — Modeling**.